In [4]:
import pandas as pd
import os
import numpy as np

# --- Main Script Logic ---

def generate_combined_stats():
    """
    Reads all labeled stock data, combines it, and generates a
    holistic statistical summary report.
    """
    try:
        # Define paths
        labeled_data_folder = "LabeledData"
        output_folder = "Summary_Reports"

        # Create the output folder if it doesn't exist
        os.makedirs(output_folder, exist_ok=True)

        # List to hold all dataframes
        all_dfs = []
        
        # Loop through all files in the LabeledData folder
        for filename in os.listdir(labeled_data_folder):
            if filename.endswith(".csv"):
                file_path = os.path.join(labeled_data_folder, filename)
                print(f"Reading {filename}...")
                df = pd.read_csv(file_path)
                all_dfs.append(df)

        if not all_dfs:
            print("No CSV files found in the LabeledData folder.")
            return

        # Concatenate all dataframes into a single one
        combined_df = pd.concat(all_dfs, ignore_index=True)
        print("Successfully combined all labeled data into a single dataframe.")
        
        # --- Per-Stock Summary Statistics ---
        
        # Group the data by 'Symbol' to get a per-stock view
        grouped_by_symbol = combined_df.groupby('Symbol')
        
        # Calculate key summary statistics for each stock
        per_stock_stats = grouped_by_symbol.agg(
            total_weeks=('Date', 'count'),
            avg_weekly_return=('Weekly_Return', 'mean'),
            std_weekly_return=('Weekly_Return', 'std'),
            avg_volume=('Total Traded Quantity', 'mean'),
        )

        # Calculate the count of 'Breakout' and 'Consolidation' labels per stock
        label_counts = grouped_by_symbol['Label'].value_counts().unstack(fill_value=0)
        
        # Join the statistics and label counts tables
        per_stock_report = per_stock_stats.join(label_counts).reset_index()
        
        print("\n--- Per-Stock Summary Report ---")
        print(per_stock_report.to_string())

        # Save the per-stock report to a CSV file
        output_path_per_stock = os.path.join(output_folder, 'per_stock_summary.csv')
        per_stock_report.to_csv(output_path_per_stock, index=False)
        print(f"\nPer-stock report saved to {output_path_per_stock}")

        # --- Overall Dataset Summary ---
        
        # Remove the 'Symbol' column to calculate stats for the entire dataset
        overall_df = combined_df.drop('Symbol', axis=1, errors='ignore')
        
        # Get a descriptive summary of the entire dataset
        overall_report = overall_df.describe(include='all')
        
        # Calculate the distribution of labels for the entire dataset
        overall_label_counts = combined_df['Label'].value_counts()
        
        # Add the label counts to the overall report
        overall_report.loc['Label_Counts'] = overall_label_counts
        
        print("\n--- Overall Dataset Summary Report ---")
        print(overall_report.to_string())

        # Save the overall report to a CSV file
        output_path_overall = os.path.join(output_folder, 'overall_dataset_summary.csv')
        overall_report.to_csv(output_path_overall, index=False)
        print(f"\nOverall dataset report saved to {output_path_overall}")

        print("\nStatistical analysis complete. Please check the 'Summary_Reports' folder for the generated files.")
        
    except FileNotFoundError as e:
        print(f"Error: The folder or file was not found. Please ensure the '{labeled_data_folder}' folder exists with the CSV files. Error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Run the main function
if __name__ == "__main__":
    generate_combined_stats()

Reading MM.csv...
Reading ICICIBANK.csv...
Reading MARUTI.csv...
Reading ASIANPAINT.csv...
Reading APOLLOHOSP.csv...
Reading HDFCBANK.csv...
Reading ADANIENT.csv...
Reading HEROMOTOCO.csv...
Reading SBIN.csv...
Reading BAJAJFINSV.csv...
Reading BAJFINANCE.csv...
Successfully combined all labeled data into a single dataframe.

--- Per-Stock Summary Report ---
        Symbol  total_weeks  avg_weekly_return  std_weekly_return    avg_volume  Breakout  Consolidation  NotImportant
0     ADANIENT          575           0.896286           8.738731  2.331018e+07         9            198           368
1   APOLLOHOSP          575           0.445193           4.211808  2.615560e+06        10             94           471
2   ASIANPAINT          575           0.329345           3.504939  6.121497e+06         9             80           486
3   BAJAJFINSV          575           0.494872           6.166078  2.708314e+06        13            136           426
4   BAJFINANCE          575           0.6416